# NB11f_R3_sensor_ablations -- R3 sensor ablation tuning (RF-200 only)

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


| pq_id | manifest_key | label | Baseline AUC |
|---|---|---|---|
| A15_R3 | `block_stats` | `card_block_stats_A15` | 0.782 |
| F8_R3 | `fusion_composite_blockstats` | `ms_plus_blocks_F8` | 0.774 |
| F7_R3 | `fusion_composite_cohdrop` | `ms_plus_cohdrop_F7` | 0.772 |
| A9_R3 | `composite_prepost_bands` | `ms_only_A9` | 0.761 |

**Source experiment:** NB09a v7 CELL R3 (verbatim CONFIG, S0 with `prepare_features`+median imputation, CLASSIFIER PARAMS including `RF_PARAMS`).

**Classifier:** RF-200 only (matches R3 source verbatim).

**Strategy:** replicate the R3 RF-200 evaluate_groupkfold call per (parquet, label), assert baseline within +/- 0.005, run Optuna with PATIENCE=40 and RESUME-IF-EXISTS SQLite. Refit best RF-200 params per ablation, persist artifacts with `mean_folds_auc`/`std_folds_auc`/`fold_aucs`.

**Output dir:** `OUTPUTS_DIR / "NB11_V2"`.

## CELL 1 -- CONFIG (verbatim NB09a v7) + NB11 additions

In [1]:
# @title CELL 3: NB09a CONFIG
TIER_SELECTION = [0,1,2]
CITY_FILTER = None
CITY_SELECTION = None
REQUIRE_UNOSAT = True
RANDOM_STATE = 42
N_FOLDS = 5

# v2 parquet system (manifest-driven). Set to False to fall back to v1 paths.
USE_V2 = True

# === NB11 ADDITIONS ===
NB11_NAME = 'NB11f_R3_sensor_ablations'
NB11_CELL_ID = 'cell_r3'

# Per-parquet config: (pq_id, manifest_key, label, baseline_auc_expected)
NB11_PARQUETS = [
    ('A15_R3', 'block_stats', 'card_block_stats_A15', 0.782),
    ('F8_R3', 'fusion_composite_blockstats', 'ms_plus_blocks_F8', 0.774),
    ('F7_R3', 'fusion_composite_cohdrop', 'ms_plus_cohdrop_F7', 0.772),
    ('A9_R3', 'composite_prepost_bands', 'ms_only_A9', 0.761),
]
NB11_BASELINE_TOL = 0.005

OPTUNA_BASE_BUDGET = 200
OPTUNA_LOW_AUC_BUDGET = 100
OPTUNA_LOW_AUC_CUTOFF = 0.70
OPTUNA_PATIENCE = 40
OPTUNA_DIRECTION = 'maximize'
OPTUNA_SEED = 42

# === THREAD BUDGET (avoid oversubscription when running parallel notebooks) ===
import os as _os, multiprocessing as _mp
NB11_PARALLEL_NOTEBOOKS = 1                                  # set to N if running N notebooks concurrently
NB11_CORES_TOTAL = _mp.cpu_count()
NB11_THREADS_PER_NB = max(1, NB11_CORES_TOTAL // NB11_PARALLEL_NOTEBOOKS)
# Env vars must be set BEFORE numpy/BLAS imports (i.e. before CELL 2 global_setup).
# Run with a fresh kernel for these to take effect at numpy import time.
_os.environ['OMP_NUM_THREADS']      = str(NB11_THREADS_PER_NB)
_os.environ['MKL_NUM_THREADS']      = str(NB11_THREADS_PER_NB)
_os.environ['OPENBLAS_NUM_THREADS'] = str(NB11_THREADS_PER_NB)
_os.environ['NUMEXPR_NUM_THREADS']  = str(NB11_THREADS_PER_NB)
_os.environ['BLIS_NUM_THREADS']     = str(NB11_THREADS_PER_NB)
print(f"[threads] cores={NB11_CORES_TOTAL}  parallel_nbs={NB11_PARALLEL_NOTEBOOKS}  "
      f"threads_per_nb={NB11_THREADS_PER_NB}")


[threads] cores=24  parallel_nbs=1  threads_per_nb=24


## CELL 1b -- enforce thread cap dynamically (BLAS/OpenMP)

Belt-and-suspenders to the env vars: if numpy/BLAS were imported on an earlier kernel state with no env-var limit, `threadpoolctl` clamps the active pools NOW without a kernel restart. Without this, LightGBM/XGBoost's OpenMP and sklearn's BLAS still spawn 24 threads each.

Set `NB11_PARALLEL_NOTEBOOKS` in CELL 1 to match the number of NB11 notebooks you'll run concurrently:
- 1 notebook  -> 24 threads per nb (no contention)
- 2 notebooks -> 12 threads each
- 3 notebooks ->  8 threads each
- 4 notebooks ->  6 threads each


In [2]:
# @title CELL 1b: dynamic thread cap via threadpoolctl
try:
    from threadpoolctl import threadpool_limits, threadpool_info
    _tp_limit = threadpool_limits(limits=NB11_THREADS_PER_NB)
    print(f"[threadpoolctl] limited BLAS/OpenMP pools to {NB11_THREADS_PER_NB} threads")
    for info in threadpool_info():
        print(f"  {info.get('prefix'):20s} ({info.get('user_api')})  "
              f"current_threads={info.get('num_threads')}  max={info.get('num_threads')}")
except ImportError:
    print("[threadpoolctl] NOT INSTALLED -- run: pip install threadpoolctl")
    print("                env vars only; may not affect pools loaded before this cell.")


[threadpoolctl] limited BLAS/OpenMP pools to 24 threads


## CELL 2 -- LOAD GLOBAL SETUP (verbatim NB09a v7)

In [3]:
# @title CELL 4: LOAD GLOBAL SETUP
import platform, os, json
if platform.system() == "Windows":
    _setup = r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks\global_setup.py"
elif os.path.exists("/content/drive_f"):
    _setup = "/content/drive_f/masterthesis/notebooks/global_setup.py"
else:
    _setup = "/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks/global_setup.py"
with open(_setup) as f:
    exec(f.read())

BDA GLOBAL SETUP
Started: 2026-05-25 07:43:03
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------


/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: True
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


<string>:564: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     911.4/7452.0 GB (6540.7 GB free)
  GDrive (F:)     1404.1/3726.0 GB (2321.9 GB free)
  Local data      11564.2/14901.9 GB (3337.7 GB free)
  Data stack      1404.1/3726.0 GB (2321.9 GB free)
  WSL ext4        69.1/1006.9 GB (886.5 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_

## CELL S0 -- MANIFEST + BUILDINGS + SHARED HELPERS (verbatim NB09a v7)

In [4]:
# @title CELL S0: LOAD MANIFEST + BUILDINGS + SHARED HELPERS
import sys, importlib, re, gc, time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, precision_recall_curve, roc_curve)
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

print("=" * 70)
print("CELL S0: NB09a v2 — LOAD MANIFEST + BUILDINGS + SHARED HELPERS")
print("=" * 70)

if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

# --- load v2 manifest -------------------------------------------------
MANIFEST_V2_PATH = DATASET_ROOT_V2 / 'parquet_manifest.json'
if not MANIFEST_V2_PATH.exists():
    raise FileNotFoundError(f"V2 manifest not found: {MANIFEST_V2_PATH}")
with open(MANIFEST_V2_PATH) as _f:
    MANIFEST = json.load(_f)
print(f"  Manifest: {MANIFEST_V2_PATH}")
print(f"  Version: {MANIFEST['version']}  Created: {MANIFEST['created']}")
print(f"  Parquets in manifest: {len(MANIFEST['parquets'])}")

# --- load building metadata (v2 path) --------------------------------
_tiers = TIER_SELECTION if isinstance(TIER_SELECTION, list) else [0, 1, 2]
_bldg_pattern = str(DATASET_ROOT_V2 / 'bda_buildings_t{tier}.parquet')
df_bldg = load_tier_parquets(_bldg_pattern, _tiers)
CITIES_TO_PROCESS, _battle_dates = resolve_cities(
    tier_selection=TIER_SELECTION,
    city_selection=[CITY_FILTER] if isinstance(CITY_FILTER, str) else CITY_FILTER,
    require_unosat=REQUIRE_UNOSAT,
)
df_bldg = df_bldg[df_bldg['city'].isin(CITIES_TO_PROCESS)].copy()
df_bldg = df_bldg[df_bldg['damage_binary'] >= 0].copy()
print(f"  Cities: {len(CITIES_TO_PROCESS)}")
print(f"  Buildings: {len(df_bldg)} (damaged={int((df_bldg['damage_binary']==1).sum())}, undamaged={int((df_bldg['damage_binary']==0).sum())})")

TARGET_COL = 'damage_binary'
EXPERIMENT_LOG = {}
DIETRICH_PAPER = {'auc': 0.813, 'f1': 0.749, 'precision': 0.671, 'recall': 0.846}

# ---- NB12 overlap-analysis: canonical OOF predictions schema ----
import hashlib as _hashlib, datetime as _dt
EXPERIMENT_ID = _dt.datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + _hashlib.md5(
    f'nb09a_{TIER_SELECTION}_{REQUIRE_UNOSAT}'.encode()).hexdigest()[:6]
print(f"  EXPERIMENT_ID: {EXPERIMENT_ID}")

def save_oof_from_result(res, model_id, oof_dir, variant_id='',
                         is_final=False, threshold=0.5):
    '''Canonical OOF schema from a result dict that contains:
       y_true, y_proba, groups, building_id, fold_id (all same length, valid-masked).'''
    if res is None:
        return None
    needed = ('y_true', 'y_proba', 'groups', 'building_id', 'fold_id')
    if not all(k in res for k in needed):
        print(f"    [save_oof] skipped {model_id}: missing keys (need {needed})")
        return None
    y_proba = np.asarray(res['y_proba'])
    yt = np.asarray(res['y_true']).astype(int)
    fold = np.asarray(res['fold_id']).astype(int)
    bid = np.asarray(res['building_id'])
    cty = np.asarray(res['groups'])
    yp = (y_proba >= threshold).astype(int)
    cm_class = np.where(yt == 1,
                        np.where(yp == 1, 'TP', 'FN'),
                        np.where(yp == 1, 'FP', 'TN'))
    oof_df = pd.DataFrame({
        'building_id':   bid,
        'city':          cty,
        'fold_id':       fold,
        'y_true':        yt,
        'y_proba':       y_proba.astype(float),
        'y_pred':        yp,
        'cm_class':      cm_class,
        'model_id':      model_id,
        'experiment_id': EXPERIMENT_ID,
        'variant_id':    variant_id,
        'is_final':      bool(is_final),
    })
    oof_dir = Path(oof_dir)
    oof_dir.mkdir(parents=True, exist_ok=True)
    out = oof_dir / f'oof_{model_id}__{EXPERIMENT_ID}.parquet'
    oof_df.to_parquet(out, index=False)
    n_tp = int((cm_class == 'TP').sum()); n_fn = int((cm_class == 'FN').sum())
    n_fp = int((cm_class == 'FP').sum()); n_tn = int((cm_class == 'TN').sum())
    print(f"    saved oof -> {out.name}  TP={n_tp} FN={n_fn} FP={n_fp} TN={n_tn}")
    return out

# --- metadata filter: single source of truth for leakage exclusion ----
# Previous hardcoded LEAKAGE_EXCLUDE_PATTERNS (4 patterns) covered only a
# subset of leakage vectors -- missing was_observed_*, s1__vv__scenes_observed,
# s2__scenes_observed, s2__lu__scenes_observed, s2__visibility__{fire,smoke,
# clear}__freq, block __count_*, rolling __count_*, and the full dietrich
# pixel-count family. metadata_filter.is_non_feature() catches all of them
# via pattern rules and a name-based metadata whitelist.
import importlib
import metadata_filter
importlib.reload(metadata_filter)
from metadata_filter import is_non_feature

def drop_leakage(cols):
    # Thin wrapper so existing prepare_features() and experiment cells keep
    # working. Strictly MORE restrictive than the old 4-pattern filter.
    return [c for c in cols if not is_non_feature(c)]

# --- Dietrich-28 stat filter (NOT a metadata leak filter) ------------
# Used by filter_dietrich28() to exclude the 'count' stat from the
# Dietrich-28 feature subset. Separate concern from metadata leakage:
# this is about matching Dietrich et al. 2025's replication spec.
EXCLUDE_CARD_STATS = ['count']

# --- dietrich column filter (from NB07 D1b, proven to work) ----------
def filter_dietrich28(cols):
    """Keep 7 backscatter stats x _mean zonal agg. Drop count (metadata)."""
    out = []
    for c in cols:
        parts = c.split('__')
        if len(parts) < 4:
            continue
        stat_agg = parts[3]
        stat_name = stat_agg.split('_')[0]
        zonal_agg = stat_agg.split('_')[-1]
        if stat_name in EXCLUDE_CARD_STATS:
            continue
        if zonal_agg != 'mean':
            continue
        out.append(c)
    return out

# --- v2 manifest-driven loader ---------------------------------------
def load_v2(manifest_key, tiers=None, columns=None, sample_frac=None):
    '''Load a v2 parquet by manifest key, merge with buildings, return df.'''
    if manifest_key not in MANIFEST['parquets']:
        raise KeyError(f"Manifest key not found: {manifest_key}")
    info = MANIFEST['parquets'][manifest_key]
    fname_tmpl = Path(info['pattern']).name
    tiers = tiers if tiers is not None else _tiers

    frames = []
    for t in tiers:
        pf = DATASET_ROOT_V2 / fname_tmpl.format(tier=t)
        if not pf.exists():
            print(f"  WARNING: {pf.name} not found, skipping tier {t}")
            continue
        if columns is not None:
            join_cols = info.get('join_keys', ['city', 'building_id'])
            read_cols = list(dict.fromkeys(join_cols + list(columns)))
            try:
                df_t = pd.read_parquet(pf, columns=read_cols)
            except Exception:
                df_t = pd.read_parquet(pf)
                keep = [c for c in read_cols if c in df_t.columns]
                df_t = df_t[keep]
        else:
            df_t = pd.read_parquet(pf)
        if sample_frac is not None and 0 < sample_frac < 1:
            df_t = df_t.sample(frac=sample_frac, random_state=RANDOM_STATE)
        frames.append(df_t)
    if not frames:
        raise FileNotFoundError(f"No v2 parquets found for key={manifest_key} tiers={tiers}")
    df = pd.concat(frames, ignore_index=True)
    del frames

    df = df.loc[:, ~df.columns.duplicated()]
    df = df[df['city'].isin(CITIES_TO_PROCESS)].copy()
    if TARGET_COL not in df.columns:
        df = df.merge(df_bldg[['building_id', 'city', TARGET_COL]].drop_duplicates(),
                      on=['building_id', 'city'], how='inner')
    df = df[df[TARGET_COL] >= 0].copy()
    df = df.loc[:, ~df.columns.duplicated()]
    print(f"  load_v2('{manifest_key}'): {len(df):,} rows, {df.shape[1]} cols, {df['city'].nunique()} cities")
    return df

# --- legacy v1 loader kept intact (some cells may still use it) ------
def load_and_merge(parquet_fmt, **kwargs):
    """V1 loader: load per-tier v1 parquets by path fmt, merge with buildings."""
    if isinstance(parquet_fmt, Path) and parquet_fmt.exists():
        df = pd.read_parquet(parquet_fmt)
    else:
        df = load_tier_parquets(str(parquet_fmt), _tiers, **kwargs)
    df = df[df['city'].isin(CITIES_TO_PROCESS)].copy()
    if TARGET_COL not in df.columns:
        df = df.merge(df_bldg[['building_id', 'city', TARGET_COL]].drop_duplicates(),
                      on=['building_id', 'city'], how='inner')
    df = df[df[TARGET_COL] >= 0].copy()
    return df

# --- feature prep + evaluation helpers (unchanged from v20) ----------
def prepare_features(df, feat_cols, drop_all_nan=True):
    """Extract X, y, groups from df, drop all-NaN columns, median-impute."""
    feat_cols = [c for c in feat_cols if c in df.columns]
    feat_cols = drop_leakage(feat_cols)
    if not feat_cols:
        print(f"    SKIP: no non-leakage feature columns")
        return None
    if drop_all_nan:
        nan_rate = df[feat_cols].isna().mean()
        feat_cols = [c for c in feat_cols if nan_rate[c] < 1.0]
    if len(feat_cols) < 2:
        print(f"    SKIP: {len(feat_cols)} feature columns after NaN drop")
        return None
    X = df[feat_cols].values
    y = df[TARGET_COL].values
    groups = df['city'].values
    building_ids = df['building_id'].values
    imp = SimpleImputer(strategy='median')
    X = imp.fit_transform(X)
    return X, y, groups, building_ids, feat_cols


def evaluate_groupkfold(clf, X, y, groups, experiment_name, n_folds=N_FOLDS, needs_scaling=False,
                        building_ids=None):
    n_cities = len(np.unique(groups))
    n_folds_actual = min(n_folds, n_cities)
    if n_folds_actual < 2:
        print(f"    SKIP {experiment_name}: only {n_cities} cities")
        return None
    gkf = GroupKFold(n_splits=n_folds_actual)
    y_proba_oof = np.full(len(y), np.nan)
    fold_id = np.full(len(y), -1, dtype=int)
    fold_aucs = []
    for fold_idx, (tr, te) in enumerate(gkf.split(X, y, groups)):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]
        if needs_scaling:
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr); Xte = scaler.transform(Xte)
        clf_c = clone(clf)
        clf_c.fit(Xtr, ytr)
        proba = clf_c.predict_proba(Xte)[:, 1] if hasattr(clf_c, 'predict_proba') else 1.0 / (1.0 + np.exp(-clf_c.decision_function(Xte)))
        y_proba_oof[te] = proba
        fold_id[te] = fold_idx
        if len(np.unique(yte)) > 1:
            fold_aucs.append(roc_auc_score(yte, proba))
    valid = ~np.isnan(y_proba_oof)
    y_v, p_v = y[valid], y_proba_oof[valid]
    pred_v = (p_v >= 0.5).astype(int)
    bid_v = np.asarray(building_ids)[valid] if building_ids is not None else np.arange(len(y))[valid].astype(str)
    result = {
        'experiment': experiment_name,
        'auc_groupkfold': roc_auc_score(y_v, p_v),
        'f1_groupkfold': f1_score(y_v, pred_v),
        'precision': precision_score(y_v, pred_v, zero_division=0),
        'recall': recall_score(y_v, pred_v, zero_division=0),
        'fold_aucs': fold_aucs,
        'auc_mean': float(np.mean(fold_aucs)) if fold_aucs else float('nan'),
        'auc_std': float(np.std(fold_aucs)) if fold_aucs else float('nan'),
        'n_features': X.shape[1], 'n_buildings': len(y), 'n_cities': len(np.unique(groups)),
        'y_true': y_v, 'y_proba': p_v, 'groups': groups[valid],
        'building_id': bid_v, 'fold_id': fold_id[valid],
    }
    EXPERIMENT_LOG[experiment_name] = result
    print(f"    {experiment_name:45s} AUC={result['auc_groupkfold']:.3f}  F1={result['f1_groupkfold']:.3f}  n={valid.sum():,} cities={result['n_cities']}")
    return result


# --- result logging / output paths -----------------------------------
import matplotlib.pyplot as plt
from datetime import datetime as _dt

OUT_DIR = OUTPUTS_DIR / "NB09a_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def save_result(data, name, cell_id, fmt='csv'):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    if fmt == 'csv' and isinstance(data, pd.DataFrame):
        path = cell_dir / f"{name}_{ts}.csv"
        data.to_csv(path, index=False)
    elif fmt == 'json':
        path = cell_dir / f"{name}_{ts}.json"
        with open(path, 'w') as fh:
            json.dump(data, fh, indent=2, default=str)
    else:
        raise ValueError(f"Unknown fmt={fmt}")
    print(f"  Saved: {path.relative_to(OUT_DIR)} ({path.stat().st_size / 1024:.1f} KB)")
    return path


def save_fig(fig, name, cell_id, dpi=150):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    path = cell_dir / f"{name}_{ts}.png"
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"  Plot: {path.relative_to(OUT_DIR)}")
    return path


from bda_results import ResultRegistry
registry = ResultRegistry(RESULTS_ROOT, notebook='NB09a_v2')


def log_result(res, cell_id, parquet_name, feature_set_name, feature_cols,
               classifier_name='RF-200', classifier_params=None, cv_method='GroupKFold',
               note='', tags=None):
    if res is None:
        return
    registry.log_experiment(
        cell_id=cell_id, experiment_name=res['experiment'],
        parquet_name=parquet_name, feature_set_name=feature_set_name,
        classifier_name=classifier_name, classifier_params=classifier_params or {},
        feature_cols=feature_cols, cv_method=cv_method, n_folds=N_FOLDS,
        imputation='median', y_true=res['y_true'], y_proba=res['y_proba'],
        groups=res['groups'], note=note, tags=tags or [],
    )


print(f"  Helpers: load_v2(), load_and_merge(), prepare_features(), evaluate_groupkfold()")
print(f"  Output:  {OUT_DIR}")
print(f"  Registry: NB09a_v2")

CELL S0: NB09a v2 — LOAD MANIFEST + BUILDINGS + SHARED HELPERS
  Manifest: /mnt/f/PROJECTS/masterthesis/data_stack/dataset/V2/parquet_manifest.json
  Version: v2  Created: 2026-04-26T16:31:26.125203
  Parquets in manifest: 37
  load_tier_parquets: 3 tiers, 907371 rows
  Cities: 21
  Buildings: 598595 (damaged=7332, undamaged=591263)
  EXPERIMENT_ID: 20260525_074423_7bcb82
  ResultRegistry: /content/drive_f/masterthesis/results/registry (run_id=20260525_074423)
  Helpers: load_v2(), load_and_merge(), prepare_features(), evaluate_groupkfold()
  Output:  /content/drive_f/masterthesis/data/outputs/NB09a_v2
  Registry: NB09a_v2


## CELL S0b -- OOF + SUMMARY HELPERS (verbatim NB09a v7)

In [5]:
# @title CELL S0b: OOF PLOT + SUMMARY HELPERS
# TP=red, TN=green, FP=pink, FN=gold
import matplotlib.pyplot as plt

CM_COLORS = {'TP': '#d62728', 'TN': '#2ca02c', 'FP': '#ff9ecb', 'FN': '#ffd700'}
CM_LABELS = {'TP': 'Destroyed (TP)', 'TN': 'Not destroyed (TN)',
             'FP': 'False positive (FP)', 'FN': 'False negative (FN)'}

# alias df_bldg -> df_buildings for plotting helper compatibility
df_buildings = df_bldg

def print_cm_summary(oof_path_or_df):
    if isinstance(oof_path_or_df, (str, Path)):
        oof = pd.read_parquet(oof_path_or_df)
    else:
        oof = oof_path_or_df
    model_id = oof['model_id'].iloc[0] if 'model_id' in oof.columns else '(unknown)'
    print(f"  model_id: {model_id}   n={len(oof)}")
    overall = oof['cm_class'].value_counts()
    for cls in ['TP', 'TN', 'FP', 'FN']:
        n = int(overall.get(cls, 0))
        pct = 100.0 * n / len(oof) if len(oof) else 0
        print(f"    {cls:3s} {CM_LABELS[cls]:28s}  n={n:6d}  ({pct:5.1f}%)")
    by_city = (oof.groupby('city')['cm_class']
                 .value_counts().unstack(fill_value=0))
    for cls in ['TP', 'TN', 'FP', 'FN']:
        if cls not in by_city.columns:
            by_city[cls] = 0
    by_city = by_city[['TP', 'TN', 'FP', 'FN']]
    by_city['recall']    = by_city['TP'] / (by_city['TP'] + by_city['FN']).replace(0, np.nan)
    by_city['precision'] = by_city['TP'] / (by_city['TP'] + by_city['FP']).replace(0, np.nan)
    print(f"\n  Per-city:")
    print(by_city.to_string(float_format=lambda x: f'{x:.3f}' if pd.notna(x) else '-'))
    return by_city

def plot_cm_spatial(oof_path_or_df, save_name=None, figsize=(14, 10),
                    buildings_df=None, point_size=4):
    if isinstance(oof_path_or_df, (str, Path)):
        oof = pd.read_parquet(oof_path_or_df)
    else:
        oof = oof_path_or_df
    bdf = buildings_df if buildings_df is not None else df_buildings
    if 'centroid_x' not in bdf.columns or 'centroid_y' not in bdf.columns:
        print("  plot_cm_spatial: no centroid_x/centroid_y in buildings_df")
        return None
    plot_df = oof.merge(bdf[['building_id', 'city', 'centroid_x', 'centroid_y']],
                        on=['building_id', 'city'], how='inner')
    if len(plot_df) == 0:
        print("  plot_cm_spatial: no building matches")
        return None
    cities = sorted(plot_df['city'].unique())
    ncols = min(3, len(cities))
    nrows = (len(cities) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    model_id = oof['model_id'].iloc[0] if 'model_id' in oof.columns else ''
    fig.suptitle(f'Confusion-matrix map: {model_id}', fontsize=11)
    draw_order = ['TN', 'FP', 'FN', 'TP']
    for i, city in enumerate(cities):
        ax = axes[i // ncols][i % ncols]
        cdf = plot_df[plot_df['city'] == city]
        for cls in draw_order:
            pts = cdf[cdf['cm_class'] == cls]
            if len(pts) == 0:
                continue
            ax.scatter(pts['centroid_x'], pts['centroid_y'],
                       c=CM_COLORS[cls], s=point_size, alpha=0.75,
                       edgecolors='none',
                       label=f"{CM_LABELS[cls]} (n={len(pts)})")
        ax.set_title(f"{city}  (n={len(cdf)})", fontsize=9)
        ax.set_aspect('equal')
        ax.legend(loc='best', fontsize=6, markerscale=2, framealpha=0.85)
        ax.tick_params(labelsize=7)
    for j in range(len(cities), nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    if save_name:
        fig.savefig(OUT_DIR / f'{save_name}.png', dpi=150, bbox_inches='tight')
        plt.close(fig)
    return fig

def plot_cm_oof_all(oof_dir, skip_variants=()):
    d = Path(oof_dir)
    files = sorted(d.glob('oof_*.parquet'))
    print(f"  Found {len(files)} OOF parquets in {d}")
    for p in files:
        oof = pd.read_parquet(p)
        v = oof['variant_id'].iloc[0] if 'variant_id' in oof.columns else ''
        if any(sv in v for sv in skip_variants):
            print(f"  skip {p.name}  (variant_id={v})")
            continue
        print(f"\n  {p.name}")
        print_cm_summary(oof)
        plot_cm_spatial(oof, save_name=p.stem)


## CELL H -- CLASSIFIER PARAMS + HELPERS (verbatim NB09a v7)

In [6]:
# @title CELL H: CLASSIFIER PARAMS + HELPERS

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

RF_PARAMS = dict(n_estimators=200, min_samples_leaf=3,
                 random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')

# sklearn GBM is single-threaded — unusable above ~100 features
GBM_MAX_FEATURES = 100

CLASSIFIERS = {
    'RF-200': RandomForestClassifier(**RF_PARAMS),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=200, min_samples_leaf=3,
                                        random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'),
    'GBM': GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                                       random_state=RANDOM_STATE),
}

try:
    from lightgbm import LGBMClassifier
    CLASSIFIERS['LightGBM'] = LGBMClassifier(n_estimators=200, max_depth=7, learning_rate=0.1,
                                              random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
                                              is_unbalance=True)
except ImportError:
    print("  LightGBM not available")

try:
    from xgboost import XGBClassifier
    CLASSIFIERS['XGBoost'] = XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.1,
                                            random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
                                            scale_pos_weight=1, use_label_encoder=False, eval_metric='logloss')
except ImportError:
    print("  XGBoost not available")

print(f"  Classifiers: {list(CLASSIFIERS.keys())}")
print(f"  GBM_MAX_FEATURES: {GBM_MAX_FEATURES} (sklearn GBM skipped above this)")

  Classifiers: ['RF-200', 'ExtraTrees', 'GBM', 'LightGBM', 'XGBoost']
  GBM_MAX_FEATURES: 100 (sklearn GBM skipped above this)


## CELL f8_feature_groups (verbatim NB09a v7 -- kept for prep-chain fidelity)

In [7]:
def f8_feature_groups(df):
    """Modality subsets of fusion_composite_blockstats F8 columns.
    Conventions:
      CARD  = s1__ (excluding s1__coh)
      COH   = s1__coh
      MS    = s2__
    """
    cols = list(df.columns)
    card = [c for c in cols if c.startswith('s1__') and not c.startswith('s1__coh')]
    coh  = [c for c in cols if c.startswith('s1__coh')]
    ms   = [c for c in cols if c.startswith('s2__')]

    groups = {
        'card_only':       drop_leakage(card),
        'coh_only':        drop_leakage(coh),
        'ms_only':         drop_leakage(ms),
        'card_coh':        drop_leakage(card + coh),
        'card_ms':         drop_leakage(card + ms),
        'coh_ms':          drop_leakage(coh + ms),
        'all_multimodal':  drop_leakage(card + coh + ms),
    }
    return groups

## CELL BASELINE -- replicate R3 RF-200 evaluate_groupkfold call per ablation

In [8]:
# @title CELL BASELINE: replicate R3 sensor ablation legs (RF-200 only)
print("=" * 70)
print(f"CELL BASELINE: {NB11_NAME} baseline replication")
print("=" * 70)

# Verbatim R3 aggregator (NB09a v7 R3 defines its own simple mean+std; agg_flag=False everywhere)
def r3_aggregate_long_to_wide(df, feat_cols):
    agg = df.groupby(['city', 'building_id'])[feat_cols].agg(['mean', 'std'])
    agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    agg = agg.reset_index()
    return agg

NB11_BASELINE_RESULTS = {}
NB11_PREP = {}

for pq_id, manifest_key, label, expected_auc in NB11_PARQUETS:
    print(f"\n  --- [{pq_id}] {label} ({manifest_key}, expected AUC={expected_auc:.4f}) ---")
    try:
        df = load_v2(manifest_key)
    except (FileNotFoundError, KeyError) as e:
        raise RuntimeError(f"load_v2 failed for {manifest_key}: {e}")

    info = MANIFEST['parquets'][manifest_key]
    feat_cols = [c for c in info['feature_columns'] if c in df.columns]

    # NB09a v7 R3: agg_flag=False for all 6 ablations -- never aggregates
    result = prepare_features(df, feat_cols)
    assert result is not None, f"prepare_features returned None for {pq_id}"
    X, y, groups, building_ids, clean = result

    rf = RandomForestClassifier(**RF_PARAMS)
    exp_name = f"R3_{label}"
    res = evaluate_groupkfold(rf, X, y, groups, exp_name, building_ids=building_ids)
    log_result(res, cell_id='cell_r3', parquet_name=f'bda_{manifest_key}_v2',
               feature_set_name=label, feature_cols=clean,
               note=f'Sensor ablation: {label}',
               tags=['sensor_ablation', label])

    got = res['auc_groupkfold']
    delta = got - expected_auc
    ok = abs(delta) <= NB11_BASELINE_TOL
    flag = 'OK' if ok else 'FAIL'
    print(f"    Replicated baseline AUC = {got:.4f}  expected = {expected_auc:.4f}  "
          f"delta = {delta:+.4f}  [{flag}]")
    if not ok:
        raise AssertionError(
            f"Baseline AUC mismatch for {pq_id}/{label}: got {got:.4f}, "
            f"expected {expected_auc:.4f} (tol {NB11_BASELINE_TOL})."
        )

    NB11_BASELINE_RESULTS[pq_id] = res
    NB11_PREP[pq_id] = {
        'X': X, 'y': y, 'groups': groups,
        'building_ids': building_ids, 'clean_cols': clean,
        'manifest_key': manifest_key, 'label': label,
        'baseline_auc': got, 'expected_auc': expected_auc,
    }

    del df
    gc.collect()

print("\n  -> All baselines match source; safe to proceed to Optuna.")


CELL BASELINE: NB11f_R3_sensor_ablations baseline replication

  --- [A15_R3] card_block_stats_A15 (block_stats, expected AUC=0.7820) ---
  load_v2('block_stats'): 598,595 rows, 1640 cols, 21 cities
    R3_card_block_stats_A15                       AUC=0.782  F1=0.101  n=598,595 cities=21
  REG: R3_card_block_stats_A15                       AUC=0.7824 F1=0.1013 n=598595 feat=780 cities=21 [NB09a_v2/cell_r3]
    Replicated baseline AUC = 0.7824  expected = 0.7820  delta = +0.0004  [OK]

  --- [F8_R3] ms_plus_blocks_F8 (fusion_composite_blockstats, expected AUC=0.7740) ---
  load_v2('fusion_composite_blockstats'): 480,313 rows, 2414 cols, 19 cities
    R3_ms_plus_blocks_F8                          AUC=0.774  F1=0.101  n=480,313 cities=19
  REG: R3_ms_plus_blocks_F8                          AUC=0.7737 F1=0.1014 n=480313 feat=915 cities=19 [NB09a_v2/cell_r3]
    Replicated baseline AUC = 0.7737  expected = 0.7740  delta = -0.0003  [OK]

  --- [F7_R3] ms_plus_cohdrop_F7 (fusion_composite_co

## CELL OPTUNA -- RF-200 tuning per ablation

In [9]:
# @title CELL OPTUNA: RF-200 search per parquet (R3 sensor ablations)
import optuna
from optuna.samplers import TPESampler
import joblib

NB11_OUT_DIR = OUTPUTS_DIR / "NB11_V2"
(NB11_OUT_DIR / "studies").mkdir(parents=True, exist_ok=True)
(NB11_OUT_DIR / "oof").mkdir(parents=True, exist_ok=True)
(NB11_OUT_DIR / "models").mkdir(parents=True, exist_ok=True)
(NB11_OUT_DIR / "best_params").mkdir(parents=True, exist_ok=True)
(NB11_OUT_DIR / "summary").mkdir(parents=True, exist_ok=True)

from sklearn.ensemble import RandomForestClassifier

def _oof_auc(clf, X, y, groups):
    gkf = GroupKFold(n_splits=min(N_FOLDS, len(np.unique(groups))))
    y_proba_oof = np.full(len(y), np.nan)
    for tr, te in gkf.split(X, y, groups):
        clf_c = clone(clf)
        clf_c.fit(X[tr], y[tr])
        y_proba_oof[te] = clf_c.predict_proba(X[te])[:, 1]
    valid = ~np.isnan(y_proba_oof)
    return roc_auc_score(y[valid], y_proba_oof[valid])

def make_rf(params):
    return RandomForestClassifier(
        random_state=RANDOM_STATE, n_jobs=NB11_THREADS_PER_NB, class_weight='balanced', **params)

def suggest_rf(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_categorical('max_depth', [3, 5, 7, 10, 15, None]),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5, 0.7]),
    }

BASELINE_TRIAL_RF = dict(n_estimators=200, max_depth=None, min_samples_leaf=3,
                         min_samples_split=2, max_features='sqrt')

def patience_stop_factory(patience):
    def cb(study, trial):
        best = study.best_trial
        if (trial.number - best.number) >= patience:
            print(f"      [patience] no improvement for {patience} trials -> stopping")
            study.stop()
    return cb

NB11_OPTUNA_RESULTS = {}
for pq_id, manifest_key, label, _expected in NB11_PARQUETS:
    prep = NB11_PREP[pq_id]
    X_pq, y_pq, groups_pq = prep['X'], prep['y'], prep['groups']
    baseline_auc = prep['baseline_auc']
    n_trials = OPTUNA_BASE_BUDGET if baseline_auc >= OPTUNA_LOW_AUC_CUTOFF else OPTUNA_LOW_AUC_BUDGET

    STUDY_TAG = f"{NB11_NAME}__{manifest_key}__{label}__RF-200"
    STUDY_DB  = NB11_OUT_DIR / "studies" / f"{STUDY_TAG}.sqlite"
    storage_url = f"sqlite:///{STUDY_DB.as_posix()}"

    print(f"\n  === [{pq_id}] {label} ({manifest_key}) ===")
    print(f"    baseline_auc = {baseline_auc:.4f}   budget = {n_trials} trials   patience = {OPTUNA_PATIENCE}")
    print(f"    DB: {STUDY_DB}")

    def objective(trial, X=X_pq, y=y_pq, g=groups_pq):
        params = suggest_rf(trial)
        clf = make_rf(params)
        return _oof_auc(clf, X, y, g)

    study = optuna.create_study(
        direction=OPTUNA_DIRECTION,
        storage=storage_url,
        study_name=STUDY_TAG,
        load_if_exists=True,
        sampler=TPESampler(seed=OPTUNA_SEED),
    )

    completed_trials = [t for t in study.trials if t.state.name == 'COMPLETE']
    if len(completed_trials) > 0:
        best = study.best_trial
        n_trials_done = len(study.trials)
        print(f"    RESUME: {n_trials_done} trials in study ({len(completed_trials)} complete) -- using as-is")
        print(f"    Best AUC: {best.value:.4f}  (trial {best.number})")
        print(f"    Best params: {best.params}")
        print(f"    Delta vs baseline: {best.value - baseline_auc:+.4f}")
    else:
        study.enqueue_trial(BASELINE_TRIAL_RF)
        print(f"    enqueued baseline params as trial 0: {BASELINE_TRIAL_RF}")
        study.optimize(objective, n_trials=n_trials,
                       callbacks=[patience_stop_factory(OPTUNA_PATIENCE)],
                       show_progress_bar=False)
        n_trials_done = len(study.trials)
        best = study.best_trial
        print(f"    Optuna done: {n_trials_done} trials")
        print(f"    Best AUC: {best.value:.4f}  (trial {best.number})")
        print(f"    Best params: {best.params}")
        print(f"    Delta vs baseline: {best.value - baseline_auc:+.4f}")

    NB11_OPTUNA_RESULTS[pq_id] = {
        'manifest_key': manifest_key,
        'label': label,
        'baseline_auc': baseline_auc,
        'tuned_auc': best.value,
        'best_params': best.params,
        'n_trials_done': n_trials_done,
        'n_trials_requested': n_trials,
        'study_tag': STUDY_TAG,
        'study_db': STUDY_DB,
        'make_clf': make_rf,
    }



  === [A15_R3] card_block_stats_A15 (block_stats) ===
    baseline_auc = 0.7824   budget = 200 trials   patience = 40
    DB: /content/drive_f/masterthesis/data/outputs/NB11_V2/studies/NB11f_R3_sensor_ablations__block_stats__card_block_stats_A15__RF-200.sqlite


[I 2026-05-25 08:09:47,302] Using an existing study with name 'NB11f_R3_sensor_ablations__block_stats__card_block_stats_A15__RF-200' instead of creating a new one.


    RESUME: 17 trials in study (16 complete) -- using as-is
    Best AUC: 0.8210  (trial 15)
    Best params: {'n_estimators': 593, 'max_depth': 10, 'min_samples_leaf': 8, 'min_samples_split': 11, 'max_features': 0.5}
    Delta vs baseline: +0.0386

  === [F8_R3] ms_plus_blocks_F8 (fusion_composite_blockstats) ===
    baseline_auc = 0.7737   budget = 200 trials   patience = 40
    DB: /content/drive_f/masterthesis/data/outputs/NB11_V2/studies/NB11f_R3_sensor_ablations__fusion_composite_blockstats__ms_plus_blocks_F8__RF-200.sqlite


[I 2026-05-25 08:09:47,591] A new study created in RDB with name: NB11f_R3_sensor_ablations__fusion_composite_blockstats__ms_plus_blocks_F8__RF-200


    enqueued baseline params as trial 0: {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 3, 'min_samples_split': 2, 'max_features': 'sqrt'}


...

      [patience] no improvement for 40 trials -> stopping
    Optuna done: 154 trials
    Best AUC: 0.8187  (trial 113)
    Best params: {'n_estimators': 314, 'max_depth': 7, 'min_samples_leaf': 17, 'min_samples_split': 20, 'max_features': 'sqrt'}
    Delta vs baseline: +0.0467

  === [A9_R3] ms_only_A9 (composite_prepost_bands) ===
    baseline_auc = 0.7612   budget = 200 trials   patience = 40
    DB: /content/drive_f/masterthesis/data/outputs/NB11_V2/studies/NB11f_R3_sensor_ablations__composite_prepost_bands__ms_only_A9__RF-200.sqlite


[I 2026-05-27 05:17:26,138] A new study created in RDB with name: NB11f_R3_sensor_ablations__composite_prepost_bands__ms_only_A9__RF-200


    enqueued baseline params as trial 0: {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 3, 'min_samples_split': 2, 'max_features': 'sqrt'}


[I 2026-05-27 05:20:57,018] Trial 0 finished with value: 0.7612317910325999 and parameters: {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 3, 'min_samples_split': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7612317910325999.
[I 2026-05-27 05:27:12,089] Trial 1 finished with value: 0.7826065793948356 and parameters: {'n_estimators': 362, 'max_depth': 3, 'min_samples_leaf': 18, 'min_samples_split': 13, 'max_features': 0.3}. Best is trial 1 with value: 0.7826065793948356.
[I 2026-05-27 05:40:08,840] Trial 2 finished with value: 0.7611826644521977 and parameters: {'n_estimators': 227, 'max_depth': None, 'min_samples_leaf': 3, 'min_samples_split': 7, 'max_features': 0.3}. Best is trial 1 with value: 0.7826065793948356.
[I 2026-05-27 05:45:38,495] Trial 3 finished with value: 0.7798532993512284 and parameters: {'n_estimators': 515, 'max_depth': None, 'min_samples_leaf': 17, 'min_samples_split': 7, 'max_features': 'log2'}. Best is trial 1 with value: 0.782606579394

      [patience] no improvement for 40 trials -> stopping
    Optuna done: 135 trials
    Best AUC: 0.7937  (trial 94)
    Best params: {'n_estimators': 764, 'max_depth': 7, 'min_samples_leaf': 18, 'min_samples_split': 19, 'max_features': 0.7}
    Delta vs baseline: +0.0324


## CELL OPTUNA-FINAL -- refit + save RF-200 per ablation with `mean_folds_auc`

In [10]:
import importlib, nb11_finalize
importlib.reload(nb11_finalize)
nb11_finalize.finalize_group_b_r3(globals())


  === refit [A15_R3] card_block_stats_A15 (block_stats) ===
    Tuned OOF AUC (refit): 0.8210  (study.best=0.8210)  delta=+0.0000
    mean(folds)=0.7077  std(folds)=0.0713
    saved oof -> oof_NB11f_R3_sensor_ablations__block_stats__card_block_stats_A15__RF-200-Optuna__20260525_074423_7bcb82.parquet  TP=5697 FN=1635 FP=113094 TN=478169
    Saved model: /content/drive_f/masterthesis/data/outputs/NB11_V2/models/NB11f_R3_sensor_ablations__block_stats__card_block_stats_A15__RF-200__best.joblib
    Saved best_params: /content/drive_f/masterthesis/data/outputs/NB11_V2/best_params/NB11f_R3_sensor_ablations__block_stats__card_block_stats_A15__RF-200__best.json
  REG: R3_card_block_stats_A15__OPTUNA               AUC=0.8210 F1=0.0903 n=598595 feat=780 cities=21 [NB09a_v2/cell_r3]

  === refit [F8_R3] ms_plus_blocks_F8 (fusion_composite_blockstats) ===
    Tuned OOF AUC (refit): 0.8130  (study.best=0.8130)  delta=+0.0000
    mean(folds)=0.7185  std(folds)=0.0806
    saved oof -> oof_NB11f_R3_se

## CELL SUMMARY -- multi-row CSV with `mean_folds_auc`/`std_folds_auc` + per-fold AUC print

In [11]:
# @title CELL SUMMARY
rows = []
for pq_id, info in NB11_OPTUNA_RESULTS.items():
    prep = NB11_PREP[pq_id]
    rows.append({
        'notebook': NB11_NAME,
        'parquet_id': pq_id,
        'manifest_key': info['manifest_key'],
        'feature_set': info['label'],
        'classifier': 'RF-200',
        'baseline_auc': info['baseline_auc'],
        'baseline_auc_expected': prep['expected_auc'],
        'tuned_auc': info['tuned_auc_refit'],
        'delta_auc': info['tuned_auc_refit'] - info['baseline_auc'],
        'mean_folds_auc': info['mean_folds_auc'],
        'std_folds_auc': info['std_folds_auc'],
        'n_trials_requested': info['n_trials_requested'],
        'n_trials_done': info['n_trials_done'],
        'early_stopped': info['n_trials_done'] < info['n_trials_requested'],
        'patience': OPTUNA_PATIENCE,
        'n_features': len(prep['clean_cols']),
        'n_buildings': len(prep['y']),
        'n_cities': int(len(np.unique(prep['groups']))),
        'study_tag': info['study_tag'],
    })
df_summary = pd.DataFrame(rows).sort_values('tuned_auc', ascending=False)
summary_path = NB11_OUT_DIR / "summary" / f"{NB11_NAME}_summary.csv"
df_summary.to_csv(summary_path, index=False)
print(df_summary.to_string(index=False))
print(f"\n  Saved summary: {summary_path}")

print("\n  Per-fold AUC (tuned models):")
for pq_id, info in NB11_OPTUNA_RESULTS.items():
    fold_aucs = info['fold_aucs']
    print(f"    [{pq_id}] {info['label']:30s} pooled={info['tuned_auc_refit']:.4f}  "
          f"mean(folds)={info['mean_folds_auc']:.4f}  std={info['std_folds_auc']:.4f}  "
          f"folds={[f'{a:.3f}' for a in fold_aucs]}")

print(f"\n  {NB11_NAME} complete.")


                 notebook parquet_id                manifest_key          feature_set classifier  baseline_auc  baseline_auc_expected  tuned_auc  delta_auc  mean_folds_auc  std_folds_auc  n_trials_requested  n_trials_done  early_stopped  patience  n_features  n_buildings  n_cities                                                                         study_tag
NB11f_R3_sensor_ablations     A15_R3                 block_stats card_block_stats_A15     RF-200      0.782387                  0.782   0.820993   0.038606        0.707713       0.071309                 200             17           True        40         780       598595        21              NB11f_R3_sensor_ablations__block_stats__card_block_stats_A15__RF-200
NB11f_R3_sensor_ablations      F7_R3    fusion_composite_cohdrop   ms_plus_cohdrop_F7     RF-200      0.771992                  0.772   0.818662   0.046670        0.741676       0.042484                 200            154           True        40         145       466704 